# COMP5339 Assignment 1 - Step 3 Notebook

This notebook extends the previous data processing pipeline by implementing Step 3: Data Augmentation.
The primary objective of this stage is to enrich the integrated facility-level dataset with geographic information. The workflow builds upon the processed outputs from Steps 1 and 2, particularly the consolidated facility-level dataset. It extracts unique facility names, performs geocoding requests, and appends the resulting spatial attributes to the dataset.

All outputs are saved into a `data_outputs/` directory located alongside this notebook. This notebook is designed to be reproducible and modular, allowing geospatial enrichment to be rerun or extended as needed.

> Note: This notebook is designed specifically for Step 3 (Data Augmentation) and builds upon the processed outputs generated in Steps 1 and 2. It assumes that the required datasets are already available in the `data_outputs/processed/` directory.  
>  
> The geocoding process relies on external API requests, so an active internet connection is required. To avoid rate limiting, requests are performed sequentially with delays.  
>  
> If required packages such as `requests`, `pandas`, or `geopy` are not installed in your environment, please install them using `requirements.txt` before running this notebook.

## 1. Imports, Configuration, and Geocoding Utilities

This cell sets up the required libraries, configuration parameters, and helper functions for the data augmentation stage. It defines file paths, API settings, request limits, and utility functions for name normalization and geocoding queries.

In [4]:
import os
import time
import requests
import pandas as pd
import duckdb


# File Paths
BASE_DIR = os.getcwd()

INPUT_PATH = os.path.join(
    BASE_DIR,
    "data_outputs",
    "processed",
    "consolidated_facility_year.csv"
)

OUTPUT_PATH = os.path.join(
    BASE_DIR,
    "data_outputs",
    "processed",
    "facility_geocoded.csv"
)

INPUT_DB = os.path.join(BASE_DIR, "data_outputs", "assignment1_steps_1_2.duckdb")
OUTPUT_DB = os.path.join(BASE_DIR, "data_outputs", "assignment1_steps_3.duckdb")


def normalize_text(text):
    if pd.isna(text):
        return None
    return str(text).strip().lower()

# API CONFIG
# Google API
GOOGLE_API_KEY = "KEY"

# OSM（fallback）
OSM_URL = "https://nominatim.openstreetmap.org/search"

HEADERS = {
    "User-Agent": "comp5339-assignment"
}

REQUEST_DELAY = 0.2  

cache = {}


# Geocoding Functions
def google_geocode(query):
    """
    Google Geocoding API
    """
    try:
        url = "https://maps.googleapis.com/maps/api/geocode/json"

        params = {
            "address": query,
            "key": GOOGLE_API_KEY
        }

        response = requests.get(url, params=params)

        if response.status_code == 200:
            data = response.json()

            if data["status"] == "OK":
                loc = data["results"][0]["geometry"]["location"]
                return loc["lat"], loc["lng"]

    except Exception as e:
        print(f"[Google Error] {query} -> {e}")

    return None, None


def osm_geocode(query):
    """
    OpenStreetMap fallback
    """
    try:
        params = {
            "q": query,
            "format": "json",
            "limit": 1
        }

        response = requests.get(OSM_URL, params=params, headers=HEADERS)

        if response.status_code == 200:
            data = response.json()

            if len(data) > 0:
                return float(data[0]["lat"]), float(data[0]["lon"])

    except Exception as e:
        print(f"[OSM Error] {query} -> {e}")

    return None, None


def cached_geocode(query):
    """
    Cache + fallback strategy
    """
    if query in cache:
        return cache[query]

    lat, lon = google_geocode(query)

    if lat is None:
        lat, lon = osm_geocode(query)

    cache[query] = (lat, lon)

    time.sleep(REQUEST_DELAY)

    return lat, lon


# Main Geocode Logic
def safe_geocode(name, state, matched_name=None):
    """
    Final optimized geocoding:
    - matched_name priority
    - cleaned queries
    - deduplication
    - cache + fallback
    """

    if pd.isna(name) and pd.isna(matched_name):
        return None, None

    # State Mapping
    STATE_MAP = {
        "act": "Australian Capital Territory",
        "nsw": "New South Wales",
        "vic": "Victoria",
        "qld": "Queensland",
        "sa": "South Australia",
        "wa": "Western Australia",
        "tas": "Tasmania",
        "nt": "Northern Territory"
    }

    state_clean = str(state).lower() if state else None
    state_full = STATE_MAP.get(state_clean, state)

    # Clean Function
    def clean(text):
        if pd.isna(text):
            return None
        text = str(text).lower()
        for w in ["lfg", "power station", "power plant", "station", "plant"]:
            text = text.replace(w, "")
        return text.strip()

    clean_name = clean(name)
    clean_matched = clean(matched_name)

    # Build Queries
    queries = []

    # matched_name
    if pd.notna(matched_name):
        queries.append(f"{matched_name}, {state_full}, Australia")
        if clean_matched:
            queries.append(f"{clean_matched}, {state_full}, Australia")

    # fallback: original name
    if pd.notna(name):
        queries.append(f"{name}, {state_full}, Australia")
        if clean_name:
            queries.append(f"{clean_name}, {state_full}, Australia")

    if clean_name:
        queries.append(f"{clean_name}, Australia")

    queries = list(dict.fromkeys(queries))

    # Execute Geocoding
    for q in queries:
        lat, lon = cached_geocode(q)
        if lat is not None:
            return lat, lon

    return None, None

## 2. Load Processed Facility-Level Dataset

This cell loads the processed facility-level dataset generated in Steps 1 and 2.

In [6]:
df = pd.read_csv(INPUT_PATH)

# Basic inspection
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (4862, 57)


,reporting_entity,facility_name,type,state,electricity_production_gj,electricity_production_mwh,total_scope_1_emissions_t_co2_e,total_scope_2_emissions_t_co2_e,total_emissions_t_co2_e,emission_intensity_t_co2_e_mwh,...,facility_match_found,matched_is_renewable_power_station,business_entries_count,business_exits_count,employee_income_earners_count,mean_employee_income_aud,median_employee_income_aud,total_businesses_count,total_employee_income_million_aud,utilities_businesses_count
0,ACCIONA ENERGY OCEANIA PTY LTD,Royalla Solar Farm,F,ACT,213115.0,59199.0,0.0,15.0,15.0,0.00,...,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ENERGY DEVELOPMENTS LIMITED,Mugga Lane LFG Power Station,F,ACT,79427.0,22063.0,1398.0,1.0,1399.0,0.06,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ENERGY DEVELOPMENTS LIMITED,West Belconnen LFG Power Station,F,ACT,11163.0,3101.0,201.0,8.0,209.0,0.06,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ROYALLA HOLDINGS ASSET PTY LTD,Royalla Solar Farm,F,ACT,49475.0,13743.0,0.0,85.0,85.0,0.00,...,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ACCIONA ENERGY OCEANIA PTY LTD,Gunning Wind Farm,F,NSW,567719.0,157700.0,19.0,293.0,312.0,0.00,...,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Extract Unique Facilities for Geocoding

To avoid redundant API calls, this step extracts the set of unique facilities based on facility name and state. This ensures that each facility is geocoded only once while preserving location-specific distinctions.

In [8]:
unique_facilities = (
    df[["facility_name", "state"]]
    .dropna(subset=["facility_name"])
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Unique facilities count:", len(unique_facilities))
unique_facilities.head()

Unique facilities count: 863


,facility_name,state
0,Royalla Solar Farm,ACT
1,Mugga Lane LFG Power Station,ACT
2,West Belconnen LFG Power Station,ACT
3,Gunning Wind Farm,NSW
4,Bayswater Power Station,NSW


### 4. Geocoding via Public APIs (Google + OpenStreetMap)

This step performs geocoding using a hybrid approach. The Google Geocoding API is used as the primary service due to its higher accuracy and coverage, while OpenStreetMap Nominatim is used as a fallback for unmatched queries.

In [10]:
results = []

print("Starting geocoding...")

for idx, row in unique_facilities.iterrows():
    name = row["facility_name"]
    state = row["state"]

    # Perform geocoding
    lat, lon = safe_geocode(
    name=row["facility_name"],
    state=row["state"],
    matched_name=row.get("matched_power_station_name", None)
)

    # Save result
    results.append({
        "facility_name": name,
        "state": state,
        "latitude": lat,
        "longitude": lon
    })

    # Progress display
    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1}/{len(unique_facilities)}")

print("Geocoding completed.")

# Convert to DataFrame
geo_df = pd.DataFrame(results)

# Preview
geo_df.head()

Starting geocoding...
Processed 50/863
Processed 100/863
Processed 150/863
Processed 200/863
Processed 250/863
Processed 300/863
Processed 350/863
Processed 400/863
Processed 450/863
Processed 500/863
Processed 550/863
Processed 600/863
Processed 650/863
Processed 700/863
Processed 750/863
Processed 800/863
Processed 850/863
Geocoding completed.


,facility_name,state,latitude,longitude
0,Royalla Solar Farm,ACT,-35.489793,149.143002
1,Mugga Lane LFG Power Station,ACT,-35.373988,149.132435
2,West Belconnen LFG Power Station,ACT,-35.240002,149.067299
3,Gunning Wind Farm,NSW,-34.690562,149.427009
4,Bayswater Power Station,NSW,-32.383841,150.952275


In [11]:
# Missing count
missing_count = geo_df["latitude"].isna().sum()

# Total count
total_count = len(geo_df)

# Percentage
missing_pct = (missing_count / total_count) * 100

print(f"Missing geocoding results: {missing_count}/{total_count} ({missing_pct:.2f}%)")
# Show failed cases
failed = geo_df[geo_df["latitude"].isna()]

failed.head(10)

Missing geocoding results: 0/863 (0.00%)


,facility_name,state,latitude,longitude


## 5. Merge Geospatial Data Back to Facility-Level Dataset

In [13]:
# Merge
df_merged = df.merge(
    geo_df,
    on=["facility_name", "state"],
    how="left"
)

# Check
print("Original rows:", len(df))
print("Merged rows:", len(df_merged))
print("Missing coordinates:", df_merged["latitude"].isna().sum())
df_merged.to_csv(OUTPUT_PATH, index=False)
print("Final dataset saved.")
df_merged.head()

Original rows: 4862
Merged rows: 4862
Missing coordinates: 0
Final dataset saved.


,reporting_entity,facility_name,type,state,electricity_production_gj,electricity_production_mwh,total_scope_1_emissions_t_co2_e,total_scope_2_emissions_t_co2_e,total_emissions_t_co2_e,emission_intensity_t_co2_e_mwh,...,business_entries_count,business_exits_count,employee_income_earners_count,mean_employee_income_aud,median_employee_income_aud,total_businesses_count,total_employee_income_million_aud,utilities_businesses_count,latitude,longitude
0,ACCIONA ENERGY OCEANIA PTY LTD,Royalla Solar Farm,F,ACT,213115.0,59199.0,0.0,15.0,15.0,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-35.489793,149.143002
1,ENERGY DEVELOPMENTS LIMITED,Mugga Lane LFG Power Station,F,ACT,79427.0,22063.0,1398.0,1.0,1399.0,0.06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-35.373988,149.132435
2,ENERGY DEVELOPMENTS LIMITED,West Belconnen LFG Power Station,F,ACT,11163.0,3101.0,201.0,8.0,209.0,0.06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-35.240002,149.067299
3,ROYALLA HOLDINGS ASSET PTY LTD,Royalla Solar Farm,F,ACT,49475.0,13743.0,0.0,85.0,85.0,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-35.489793,149.143002
4,ACCIONA ENERGY OCEANIA PTY LTD,Gunning Wind Farm,F,NSW,567719.0,157700.0,19.0,293.0,312.0,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-34.690562,149.427009


## 6. Export Augmented Dataset

In this step, the final dataset with geographic coordinates is written into the existing DuckDB database.

This updates the database with the augmented facility-level data for further querying and analysis.

In [15]:
conn = duckdb.connect(OUTPUT_DB)

try:
    conn.execute("DETACH old_db")
except:
    pass

conn.execute(f"ATTACH '{INPUT_DB}' AS old_db")
conn.execute("CREATE OR REPLACE TABLE abs_measure_coverage AS SELECT * FROM old_db.abs_measure_coverage")
conn.execute("CREATE OR REPLACE TABLE abs_state_measures AS SELECT * FROM old_db.abs_state_measures")
conn.execute("CREATE OR REPLACE TABLE cer_power_stations AS SELECT * FROM old_db.cer_power_stations")
conn.execute("CREATE OR REPLACE TABLE nger_facilities AS SELECT * FROM old_db.nger_facilities")
conn.execute("CREATE OR REPLACE TABLE integrated_state_year AS SELECT * FROM old_db.integrated_state_year")
conn.execute("CREATE OR REPLACE TABLE consolidated_facility_year AS SELECT * FROM df_merged")
conn.execute("CHECKPOINT")
conn.close()

conn = duckdb.connect(OUTPUT_DB)
tables = conn.execute("SHOW TABLES").fetchdf()
for table in tables["name"]:
    print(f"\n--- {table} ---")
    schema = conn.execute(f"DESCRIBE {table}").fetchdf()
    print(schema)
conn.execute("CHECKPOINT")
conn.close()


--- abs_measure_coverage ---
            column_name column_type null   key default extra
0          measure_code     VARCHAR  YES  None    None  None
1          measure_name     VARCHAR  YES  None    None  None
2    parent_description     VARCHAR  YES  None    None  None
3           description     VARCHAR  YES  None    None  None
4  first_available_year      BIGINT  YES  None    None  None
5   last_available_year      BIGINT  YES  None    None  None
6  available_year_count      BIGINT  YES  None    None  None
7           state_count      BIGINT  YES  None    None  None
8   expected_year_count      BIGINT  YES  None    None  None
9        coverage_ratio      DOUBLE  YES  None    None  None

--- abs_state_measures ---
          column_name column_type null   key default extra
0        measure_code     VARCHAR  YES  None    None  None
1  parent_description     VARCHAR  YES  None    None  None
2         description     VARCHAR  YES  None    None  None
3  abs_reference_year      BIGINT  